# Aliases, formulas and related aggregates

Name paths with aliasColumn, declare row, invoice, customer and product totals with formulaColumn, aggregate related records, and filter products by average selling price.

[Watch lesson 4](https://asqueel.org/learn/04-keys-and-constraints/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## Recreate the starting point

The previous lessons are rebuilt below. The four fictional CSV files are included in this notebook and written into a new temporary folder. No downloads or existing database are needed.

In [ ]:
#@title Prepare the sample CSV files
from pathlib import Path
import os
import tempfile
_lesson_folder = tempfile.TemporaryDirectory(prefix="asqueel-notebook-")
os.chdir(_lesson_folder.name)
_csv_files = {'customers.csv': 'id,name,city\n1,Ada,Florence\n2,Grace,London\n3,Linus,Helsinki\n4,Guido,Amsterdam\n5,Elena,Florence\n6,Alan,Manchester\n7,Margaret,New York\n8,Barbara,Boston\n', 'invoice_rows.csv': 'id,invoice_id,product_code,quantity,unit_price,total\n1,101,TEA,2,3.50,7.00\n2,101,COF,1,6.90,6.90\n3,101,MUG,4,5.25,21.00\n4,102,BTEA,2,3.20,6.40\n5,102,HTEA,3,4.10,12.30\n6,102,HON,4,7.50,30.00\n7,103,SUG,3,2.30,6.90\n8,103,CHO,4,3.90,15.60\n9,103,BIS,1,2.80,2.80\n10,104,JAM,4,4.60,18.40\n11,104,OAT,1,2.40,2.40\n12,104,JUI,2,2.70,5.40\n13,105,WAT,1,0.90,0.90\n14,105,LEM,2,1.80,3.60\n15,105,KET,3,32.90,98.70\n16,106,POT,2,18.50,37.00\n17,106,PLT,3,6.40,19.20\n18,106,BWL,4,7.80,31.20\n19,107,SPO,3,9.50,28.50\n20,107,TRY,4,14.90,59.60\n21,107,JAR,1,8.20,8.20\n22,108,NAP,4,12.00,48.00\n23,108,TWL,1,5.60,5.60\n24,108,FLT,2,2.10,4.20\n25,109,TIN,1,6.70,6.70\n26,109,MAT,2,11.40,22.80\n27,109,CAF,3,29.90,89.70\n28,110,BOT,2,13.50,27.00\n29,110,STR,3,4.80,14.40\n30,110,SCL,4,24.90,99.60\n31,111,TEA,3,3.50,10.50\n32,111,COF,4,6.90,27.60\n33,111,MUG,1,5.25,5.25\n34,112,BTEA,4,3.20,12.80\n35,112,HTEA,1,4.10,4.10\n36,112,HON,2,7.50,15.00\n', 'invoices.csv': 'id,number,customer_id,invoice_date\n101,INV-001,1,2026-09-28\n102,INV-002,2,2026-09-29\n103,INV-003,3,2026-09-30\n104,INV-004,4,2026-10-01\n105,INV-005,5,2026-10-02\n106,INV-006,6,2026-10-03\n107,INV-007,7,2026-10-04\n108,INV-008,8,2026-10-05\n109,INV-009,1,2026-10-06\n110,INV-010,2,2026-10-07\n111,INV-011,5,2026-10-08\n112,INV-012,3,2026-10-09\n', 'products.csv': 'code,name,category,stock\nBTEA,Black tea,Drinks,30\nHTEA,Herbal tea,Drinks,18\nHON,Honey,Food,22\nSUG,Cane sugar,Food,40\nCHO,Dark chocolate,Food,35\nBIS,Biscuits,Food,28\nJAM,Strawberry jam,Food,16\nOAT,Oat drink,Drinks,32\nJUI,Apple juice,Drinks,26\nWAT,Mineral water,Drinks,60\nLEM,Lemonade,Drinks,30\nKET,Kettle,Kitchen,9\nPOT,Teapot,Kitchen,14\nPLT,Plate,Kitchen,36\nBWL,Bowl,Kitchen,24\nSPO,Spoon set,Kitchen,20\nTRY,Serving tray,Kitchen,11\nJAR,Storage jar,Kitchen,25\nNAP,Cotton napkins,Kitchen,18\nTWL,Tea towel,Kitchen,30\nFLT,Coffee filters,Kitchen,45\nTIN,Tea tin,Kitchen,15\nMAT,Table mats,Kitchen,16\nCAF,Coffee maker,Kitchen,7\nBOT,Water bottle,Kitchen,21\nSTR,Tea strainer,Kitchen,23\nSCL,Kitchen scale,Kitchen,8\n'}
for _name, _text in _csv_files.items():
    _ = Path(_name).write_text(_text, encoding="utf-8")


In [ ]:
#@title Recreate the previous lessons
from asqueel import AsqueelDb, SqlDatabaseConfig

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")

db = AsqueelDb(model)
print(db.migrate().commands)

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()

products.query(columns="$code, $name").fetch()

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()

products.query(columns="$code, $name").fetch()
db.commit()

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")

db.reloadModel(model)

print(db.migrate().commands)

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()

products.query(columns="$name, $category, $stock").fetch()
db.commit()

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)

products.sql_insert(data, format="csv")
db.commit()

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()

# Continue with lesson two.
products.query(columns="$code, $name").fetch()
db.commit()

customer = shop.table("customer", pkey="id")
_ = customer.column("id", dtype="I")
_ = customer.column("name", size=":60")
_ = customer.column("city", size=":60")

db.reloadModel(model)
print(db.migrate().commands)

invoice = shop.table("invoice", pkey="id")
_ = invoice.column("id", dtype="I")
_ = invoice.column("number", size=":20")
_ = invoice.column("invoice_date", dtype="D")

customer_id = invoice.column("customer_id", dtype="I")
_ = customer_id.relation(
    "shop.customer.id", foreign_key=True, relation_name="invoices",
)

invoice_row = shop.table("invoice_row", pkey="id")
_ = invoice_row.column("id", dtype="I")
_ = invoice_row.column("quantity", dtype="I")
_ = invoice_row.column("unit_price", dtype="N", size="12,2")
_ = invoice_row.column("total", dtype="N", size="12,2")

invoice_id = invoice_row.column("invoice_id", dtype="I")
_ = invoice_id.relation(
    "shop.invoice.id", foreign_key=True, relation_name="rows",
)
product_code = invoice_row.column("product_code", size=":10")
_ = product_code.relation(
    "shop.product.code", foreign_key=True, relation_name="rows",
)

db.reloadModel(model)
print(db.migrate().commands)

products = db.table("shop.product")
customers = db.table("shop.customer")
invoices = db.table("shop.invoice")
rows = db.table("shop.invoice_row")

from pathlib import Path
print("\n".join(Path("invoice_rows.csv").read_text().splitlines()[:4]))

with open("products.csv", newline="", encoding="utf-8") as source:
    products.sql_insert(source)

db.commit()

products.query().count()
db.commit()

with open("customers.csv", newline="", encoding="utf-8") as source:
    customers.sql_insert(source)

db.commit()
customers.query(columns="$name, $city", limit=3).fetch()
db.commit()

with open("invoices.csv", newline="", encoding="utf-8") as source:
    invoices.sql_insert(source)

db.commit()

with open("invoice_rows.csv", newline="", encoding="utf-8") as source:
    rows.sql_insert(source)

db.commit()

invoices.query().count()
rows.query().count()
db.commit()

invoices.query(
    columns="$number, $invoice_date, "
            "@customer_id.name, @customer_id.city",
    limit=3,
).fetch()
db.commit()

rows.query(
    columns="@product_code.name, "
            "$quantity, $unit_price, $total",
    where="$invoice_id = :invoice", invoice=101,
).fetch()
db.commit()

rows.query(
    columns="@invoice_id.number, "
            "@invoice_id.@customer_id.city, "
            "@product_code.name, $total",
    limit=3,
).fetch()
db.commit()

# Continue with lesson three.
invoices.query(
    columns="$number",
    where="@customer_id.city = :city", city="Florence",
).fetch()

invoices.query(
    columns="$number, @customer_id.city",
    order_by="@customer_id.city", limit=4,
).fetch()

rows.query(
    columns="@invoice_id.invoice_date, $total",
    order_by="@invoice_id.invoice_date DESC", limit=3,
).fetch()

rows.query(
    columns="@product_code.name, $quantity",
    where="@invoice_id.@customer_id.city = :city",
    city="Florence", limit=3,
).fetch()

from datetime import date
rows.query(
    columns="@product_code.name, $total",
    where="@invoice_id.@customer_id.city = :city "
          "AND @invoice_id.invoice_date >= :since",
    city="Florence", since=date(2026, 10, 1),
    order_by="@invoice_id.invoice_date DESC", limit=4,
).fetch()

db.commit()


## 1 — What we'll learn in this lesson

Welcome back! Here's what we'll learn in this lesson. Name a relationship path, calculate row, invoice, customer and product totals, then sum, count and average related records. Finally, find products within an average selling price range.

```text
What we'll learn in this lesson
aliasColumn → name a path
formulaColumn → row, invoice, customer and product totals
@relation.aggr(...) → sum, count and average
Filter products by average selling price

Same Python session · SQLite in memory
```

The setup above has recreated the previous lessons in this notebook's kernel. Continue with the cells below.

## 2 — Give the city path a short name

Let's call this path city. Alias column gives our two-hop customer city a reusable name. Reload the model and get a fresh table handle.

In [ ]:
_ = invoice_row.aliasColumn(
    "city", relation_path="@invoice_id.@customer_id.city",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")

Let's read and filter with that name. Dollar city follows the original relationship path. It does not store another copy of the city.

In [ ]:
rows.query(
    columns="$city, @product_code.name",
    where="$city = :city", city="Florence", limit=3,
).fetch()
db.commit()

## 3 — Calculate the amount of each row

Let's calculate quantity times price. Formula column declares an SQL expression. Round it to two decimal places, and describe the result as numeric.

In [ ]:
_ = invoice_row.formulaColumn(
    "calculated_total",
    sql_formula="ROUND($quantity * $unit_price, 2)",
    dtype="N",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")

Let's read the calculated amounts. Two teas at three fifty make seven. The database evaluates the formula when we query it.

In [ ]:
rows.query(
    columns="$quantity, $unit_price, $calculated_total",
    where="$invoice_id = :invoice", invoice=101,
    order_by="$id",
).fetch()
db.commit()

## 4 — Give every invoice its total

Let's give each invoice a calculated total. Declare the calculation once in the model. Select its rows and add their calculated amounts. THIS id is the current invoice's key.

In [ ]:
_ = invoice.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$invoice_id = #THIS.id",
    ),
)
db.reloadModel(model)
invoices = db.table("shop.invoice")

Let's read those totals as ordinary fields. The query only asks for number and total. The calculation stays in the model.

In [ ]:
invoices.query(
    columns="$number, $total", order_by="$id", limit=3,
).fetch()
db.commit()

## 5 — Give every customer their total

Let's total the invoices of each customer. Reuse the invoice total we just declared. THIS id now belongs to the customer.

In [ ]:
_ = customer.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice",
        columns="ROUND(SUM($total), 2)",
        where="$customer_id = #THIS.id",
    ),
)
db.reloadModel(model)
customers = db.table("shop.customer")

Let's read each customer's sales total. Ada has two invoices. Together they make one hundred fifty-four ten.

In [ ]:
customers.query(
    columns="$name, $total", order_by="$id", limit=3,
).fetch()
db.commit()

## 6 — Give every product its sales total

Let's total the sales of each product. Add the amounts of invoice rows carrying its code. This is sales revenue, not the quantity in stock.

In [ ]:
_ = product.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")

Let's read the product totals. Again, the query uses a named field. Products with no sales have a null total.

In [ ]:
products.query(
    columns="$code, $name, $total",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()

## 7 — No new stored columns

Let's check whether the database needs changing. True means no migration is needed. Aliases and calculated totals add no physical columns.

In [ ]:
db.migration_plan().empty

## 8 — Watch the formula respond to a change

Let's temporarily change the quantity to three. The calculated amount becomes ten fifty immediately. The old total imported from CSV stays seven because it is an ordinary stored column.

In [ ]:
rows.update({"quantity": 3}, pkey=1)
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=1,
).fetch()

Let's read the three totals through the relationships. Invoice, customer and product totals all include the extra tea immediately. We did not write any of those totals.

In [ ]:
rows.query(
    columns="@invoice_id.total, "
            "@invoice_id.@customer_id.total, @product_code.total",
    where="$id = :id", id=1,
).fetch()

Let's undo that change. Rollback restores the original quantity, so the rest of our examples use the same dataset.

In [ ]:
db.rollback()

## 9 — Sum, count and average the related invoices

Now let's ask for several aggregates at once. A customer reaches its invoices through the inverse name declared in lesson two. Each customer still produces one result row.

```text
customer → @invoices
invoice  → @rows
product  → @rows

sum($total) → sales amount
count()    → number of invoices
avg($total) → average invoice amount
```

Let's summarize each customer's invoices. Aggr adds their totals, counts the invoices, and averages their amounts. Asqueel names the returned columns automatically.

In [ ]:
customers = db.table("shop.customer")
customers.query(
    columns="$name, @invoices.aggr("
            "sum($total), count(), avg($total))",
    order_by="$id", limit=3,
).fetch()
db.commit()

Let's count the rows of each invoice too. The same aggregation syntax follows rows. Count returns the number of invoice rows, not the sum of their quantities.

In [ ]:
invoices = db.table("shop.invoice")
invoices.query(
    columns="$number, $total, @rows.aggr(count())",
    order_by="$id", limit=3,
).fetch()
db.commit()

## 10 — Average the selling prices of each product

Let's average the recorded selling prices. Follow a product's invoice rows and average their unit prices. This is one price per row, without weighting by quantity.

In [ ]:
products.query(
    columns="$code, @rows.aggr(count(), avg($unit_price))",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()

## 11 — Filter products by their average selling price

Let's name the average price so we can filter it. Define the average in the model. In this release, aggr returns columns; a named formula also works in where and order by.

In [ ]:
_ = product.formulaColumn(
    "average_price", dtype="N",
    select=dict(
        table="shop.invoice_row", columns="AVG($unit_price)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")

Let's find products averaging between three twenty and four eighty. Pass both bounds as parameters. Between includes the endpoints. Sort by average price, and read the six matching products.

In [ ]:
products.query(
    columns="$code, $name, $average_price",
    where="$average_price BETWEEN :min_price AND :max_price",
    min_price=3.2, max_price=4.8,
    order_by="$average_price, $code",
).fetch()
db.commit()

## 12 — Thank you — names, formulas and related aggregates

Thanks for following along! We named paths, calculated totals in the model, and summed, counted and averaged related rows. We also filtered products by their average selling price. Next, we will explore virtual relations.

```text
Thank you!
aliasColumn → name a path
formulaColumn → reusable totals and average price
@relation.aggr(...) → sum, count and average
where → products within an average-price range

Next: virtual relations
```

## Check your result

Run this final cell to check the exercise.

In [ ]:
from decimal import Decimal
from math import isclose
assert rows.query().count() == 36
amounts = rows.query(columns="$quantity, $unit_price, $calculated_total").fetch()
assert all(Decimal(str(r["calculated_total"])).quantize(Decimal(".01")) == (r["quantity"] * r["unit_price"]).quantize(Decimal(".01")) for r in amounts)
assert rows.query(columns="$quantity", where="$id = :id", id=1).fetch()[0]["quantity"] == 2
expected_invoices = {'101': {'total': 34.9, 'count': 3, 'average': 11.633333333333333}, '102': {'total': 48.7, 'count': 3, 'average': 16.233333333333334}, '103': {'total': 25.3, 'count': 3, 'average': 8.433333333333334}, '104': {'total': 26.2, 'count': 3, 'average': 8.733333333333333}, '105': {'total': 103.2, 'count': 3, 'average': 34.4}, '106': {'total': 87.4, 'count': 3, 'average': 29.133333333333333}, '107': {'total': 96.3, 'count': 3, 'average': 32.1}, '108': {'total': 57.8, 'count': 3, 'average': 19.266666666666666}, '109': {'total': 119.2, 'count': 3, 'average': 39.733333333333334}, '110': {'total': 141.0, 'count': 3, 'average': 47.0}, '111': {'total': 43.35, 'count': 3, 'average': 14.45}, '112': {'total': 31.9, 'count': 3, 'average': 10.633333333333333}}
expected_customers = {'1': {'total': 154.1, 'count': 2, 'average': 77.05}, '2': {'total': 189.7, 'count': 2, 'average': 94.85}, '3': {'total': 57.2, 'count': 2, 'average': 28.6}, '4': {'total': 26.2, 'count': 1, 'average': 26.2}, '5': {'total': 146.55, 'count': 2, 'average': 73.275}, '6': {'total': 87.4, 'count': 1, 'average': 87.4}, '7': {'total': 96.3, 'count': 1, 'average': 96.3}, '8': {'total': 57.8, 'count': 1, 'average': 57.8}}
expected_products = {'BTEA': {'total': 19.2, 'count': 2, 'average': 9.6}, 'HTEA': {'total': 16.4, 'count': 2, 'average': 8.2}, 'HON': {'total': 45.0, 'count': 2, 'average': 22.5}, 'SUG': {'total': 6.9, 'count': 1, 'average': 6.9}, 'CHO': {'total': 15.6, 'count': 1, 'average': 15.6}, 'BIS': {'total': 2.8, 'count': 1, 'average': 2.8}, 'JAM': {'total': 18.4, 'count': 1, 'average': 18.4}, 'OAT': {'total': 2.4, 'count': 1, 'average': 2.4}, 'JUI': {'total': 5.4, 'count': 1, 'average': 5.4}, 'WAT': {'total': 0.9, 'count': 1, 'average': 0.9}, 'LEM': {'total': 3.6, 'count': 1, 'average': 3.6}, 'KET': {'total': 98.7, 'count': 1, 'average': 98.7}, 'POT': {'total': 37.0, 'count': 1, 'average': 37.0}, 'PLT': {'total': 19.2, 'count': 1, 'average': 19.2}, 'BWL': {'total': 31.2, 'count': 1, 'average': 31.2}, 'SPO': {'total': 28.5, 'count': 1, 'average': 28.5}, 'TRY': {'total': 59.6, 'count': 1, 'average': 59.6}, 'JAR': {'total': 8.2, 'count': 1, 'average': 8.2}, 'NAP': {'total': 48.0, 'count': 1, 'average': 48.0}, 'TWL': {'total': 5.6, 'count': 1, 'average': 5.6}, 'FLT': {'total': 4.2, 'count': 1, 'average': 4.2}, 'TIN': {'total': 6.7, 'count': 1, 'average': 6.7}, 'MAT': {'total': 22.8, 'count': 1, 'average': 22.8}, 'CAF': {'total': 89.7, 'count': 1, 'average': 89.7}, 'BOT': {'total': 27.0, 'count': 1, 'average': 27.0}, 'STR': {'total': 14.4, 'count': 1, 'average': 14.4}, 'SCL': {'total': 99.6, 'count': 1, 'average': 99.6}, 'TEA': {'total': 17.5, 'count': 2, 'average': 8.75}, 'COF': {'total': 34.5, 'count': 2, 'average': 17.25}, 'MUG': {'total': 26.25, 'count': 2, 'average': 13.125}}
expected_prices = {'BTEA': {'total': 6.4, 'count': 2, 'average': 3.2}, 'HTEA': {'total': 8.2, 'count': 2, 'average': 4.1}, 'HON': {'total': 15.0, 'count': 2, 'average': 7.5}, 'SUG': {'total': 2.3, 'count': 1, 'average': 2.3}, 'CHO': {'total': 3.9, 'count': 1, 'average': 3.9}, 'BIS': {'total': 2.8, 'count': 1, 'average': 2.8}, 'JAM': {'total': 4.6, 'count': 1, 'average': 4.6}, 'OAT': {'total': 2.4, 'count': 1, 'average': 2.4}, 'JUI': {'total': 2.7, 'count': 1, 'average': 2.7}, 'WAT': {'total': 0.9, 'count': 1, 'average': 0.9}, 'LEM': {'total': 1.8, 'count': 1, 'average': 1.8}, 'KET': {'total': 32.9, 'count': 1, 'average': 32.9}, 'POT': {'total': 18.5, 'count': 1, 'average': 18.5}, 'PLT': {'total': 6.4, 'count': 1, 'average': 6.4}, 'BWL': {'total': 7.8, 'count': 1, 'average': 7.8}, 'SPO': {'total': 9.5, 'count': 1, 'average': 9.5}, 'TRY': {'total': 14.9, 'count': 1, 'average': 14.9}, 'JAR': {'total': 8.2, 'count': 1, 'average': 8.2}, 'NAP': {'total': 12.0, 'count': 1, 'average': 12.0}, 'TWL': {'total': 5.6, 'count': 1, 'average': 5.6}, 'FLT': {'total': 2.1, 'count': 1, 'average': 2.1}, 'TIN': {'total': 6.7, 'count': 1, 'average': 6.7}, 'MAT': {'total': 11.4, 'count': 1, 'average': 11.4}, 'CAF': {'total': 29.9, 'count': 1, 'average': 29.9}, 'BOT': {'total': 13.5, 'count': 1, 'average': 13.5}, 'STR': {'total': 4.8, 'count': 1, 'average': 4.8}, 'SCL': {'total': 24.9, 'count': 1, 'average': 24.9}, 'TEA': {'total': 7.0, 'count': 2, 'average': 3.5}, 'COF': {'total': 13.8, 'count': 2, 'average': 6.9}, 'MUG': {'total': 10.5, 'count': 2, 'average': 5.25}}
for table, key, expected in (("shop.invoice", "id", expected_invoices), ("shop.customer", "id", expected_customers), ("shop.product", "code", expected_products)):
    result = db.table(table).query(columns=f"${key}, $total").fetch()
    assert len(result) == len(expected)
    for row in result:
        total = expected[str(row[key])]["total"]
        assert row["total"] is None if total is None else isclose(float(row["total"]), total)
report = db.table("shop.customer").query(columns="$id, @invoices.aggr(sum($total), count(), avg($total))").fetch()
assert len(report) == 8
for row in report:
    expected = expected_customers[str(row["id"])]
    assert row["_invoices_count"] == expected["count"]
    assert isclose(float(row["_invoices_total_sum"]), expected["total"])
    assert isclose(float(row["_invoices_total_avg"]), expected["average"])
report = products.query(columns="$code, $average_price, @rows.aggr(count(), avg($unit_price))").fetch()
assert len(report) == 30
for row in report:
    expected = expected_prices[row["code"]]
    assert row["_rows_count"] == expected["count"]
    for mean in (row["average_price"], row["_rows_unit_price_avg"]):
        assert mean is None if expected["average"] is None else isclose(float(mean), expected["average"])
report = products.query(columns="$code, $average_price", where="$average_price BETWEEN :min_price AND :max_price", min_price=3.2, max_price=4.8, order_by="$average_price, $code").fetch()
assert sorted(row["code"] for row in report) == ['BTEA', 'CHO', 'HTEA', 'JAM', 'STR', 'TEA']
assert isclose(float(report[0]["average_price"]), 3.2)
assert isclose(float(report[-1]["average_price"]), 4.8)
db.commit()
products.insert({"code": "CHECK", "name": "Unsold test product"})
empty = products.query(columns="$total, $average_price, @rows.aggr(count(), sum($calculated_total), avg($unit_price))", where="$code = :code", code="CHECK").fetch()[0]
assert empty["_rows_count"] == 0
assert all(empty[key] is None for key in ("total", "average_price", "_rows_calculated_total_sum", "_rows_unit_price_avg"))
db.rollback()
assert products.query().count() == 30
db.commit()
assert db.migration_plan().empty
db.commit()
print("Lesson checks passed.")
